In [16]:
import pandas as pd                                                                  
                                                                                       
  # Load the two core DataFrames
deliveries = pd.read_csv("../data/processed/deliveries.csv")                         
matches = pd.read_csv("../data/processed/matches.csv")
                                          
print(f"Deliveries: {deliveries.shape[0]:,} rows, {deliveries.shape[1]} columns")
print(f"Matches: {matches.shape[0]:,} rows, {matches.shape[1]} columns")

Deliveries: 279,586 rows, 35 columns
Matches: 1,175 rows, 14 columns


In [17]:
# Check column names and data types for deliveries
print("=== DELIVERIES SCHEMA ===")                                                   
print(deliveries.dtypes) 

=== DELIVERIES SCHEMA ===
match_id                int64
season                  int64
venue                     str
city                      str
innings                 int64
batting_team              str
bowling_team              str
over                    int64
ball_in_over            int64
batter                    str
non_striker               str
bowler                    str
batting_position        int64
target_runs           float64
target_overs          float64
super_over               bool
batter_runs             int64
extra_runs              int64
total_runs              int64
is_wide                  bool
is_noball                bool
is_boundary_4            bool
is_boundary_6            bool
is_dot                   bool
wicket                   bool
wicket_kind               str
player_out                str
fielders                  str
phase                     str
cumulative_runs         int64
cumulative_wickets      int64
match_winner              str
current_run_ra

In [18]:
# Count nulls per column — only show columns that have at least one null
null_counts = deliveries.isnull().sum()                                              
print("=== NULL COUNTS (columns with nulls only) ===")
print(null_counts[null_counts > 0]) 

=== NULL COUNTS (columns with nulls only) ===
city                  12397
target_runs          145053
target_overs         145053
wicket_kind          265685
player_out           265685
fielders             269510
match_winner           4702
current_run_rate        153
required_run_rate    146143
run_rate_pressure    147721
dtype: int64


In [19]:
# How many unique matches have no winner recorded?
null_winner_matches = deliveries[deliveries["match_winner"].isnull()]["match_id"].nunique()                
print(f"Matches with no winner: {null_winner_matches}")

Matches with no winner: 23


In [20]:
# Check value ranges for key numerical columns
print("=== VALUE RANGES ===")                                                        
print(f"over:          {deliveries['over'].min()} to {deliveries['over'].max()}")
print(f"ball_in_over:  {deliveries['ball_in_over'].min()} to {deliveries['ball_in_over'].max()}")        
print(f"batter_runs:   {deliveries['batter_runs'].min()} to {deliveries['batter_runs'].max()}")                                                  
print(f"total_runs:    {deliveries['total_runs'].min()} to {deliveries['total_runs'].max()}")                                                   
print(f"extra_runs:    {deliveries['extra_runs'].min()} to {deliveries['extra_runs'].max()}")
print(f"innings:       {deliveries['innings'].unique()}")                            
print(f"phase:         {deliveries['phase'].unique()}")

=== VALUE RANGES ===
over:          1 to 20
ball_in_over:  0 to 7
batter_runs:   0 to 6
total_runs:    0 to 7
extra_runs:    0 to 7
innings:       [1 2 3 4 5 6]
phase:         <ArrowStringArray>
['powerplay', 'middle', 'death']
Length: 3, dtype: str


In [21]:
# How many deliveries and matches have innings > 2?                                
high_innings = deliveries[deliveries["innings"] > 2]
print(f"Deliveries with innings > 2: {len(high_innings)}")                           
print(f"Unique matches:              {high_innings['match_id'].nunique()}")
print(f"Innings values seen:         {sorted(high_innings['innings'].unique())}")    
print(f"Super over flag on these:    {high_innings['super_over'].unique()}")

Deliveries with innings > 2: 171
Unique matches:              15
Innings values seen:         [np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Super over flag on these:    [ True]


In [22]:
# Separate innings 1 and innings 2 (exclude super overs)                             
inn1 = deliveries[(deliveries["innings"] == 1) & (deliveries["super_over"] == False)]
inn2 = deliveries[(deliveries["innings"] == 2) & (deliveries["super_over"] == False)]
                                            
print("=== 1ST INNINGS ===")                                                         
print(f"current_run_rate nulls:   {inn1['current_run_rate'].isnull().sum()}")
print(f"required_run_rate nulls:  {inn1['required_run_rate'].isnull().sum()}  (should = {len(inn1)})")                           
print(f"run_rate_pressure nulls:  {inn1['run_rate_pressure'].isnull().sum()}  (should = {len(inn1)})")                                                                    
                                                                                    
print("\n=== 2ND INNINGS ===")                                                       
print(f"current_run_rate nulls:   {inn2['current_run_rate'].isnull().sum()}")        
print(f"required_run_rate nulls:  {inn2['required_run_rate'].isnull().sum()}")
print(f"run_rate_pressure nulls:  {inn2['run_rate_pressure'].isnull().sum()}")       
print(f"current_run_rate range:   {inn2['current_run_rate'].min():.2f} to {inn2['current_run_rate'].max():.2f}")                                               
print(f"required_run_rate range:  {inn2['required_run_rate'].min():.2f} to {inn2['required_run_rate'].max():.2f}")                                              
print(f"run_rate_pressure range:  {inn2['run_rate_pressure'].min():.2f} to {inn2['run_rate_pressure'].max():.2f}")

=== 1ST INNINGS ===
current_run_rate nulls:   73
required_run_rate nulls:  144882  (should = 144882)
run_rate_pressure nulls:  144882  (should = 144882)

=== 2ND INNINGS ===
current_run_rate nulls:   78
required_run_rate nulls:  1090
run_rate_pressure nulls:  2668
current_run_rate range:   0.00 to 60.00
required_run_rate range:  0.07 to 36.00
run_rate_pressure range:  0.00 to 23.51


In [23]:
print("=== VALIDATION SUMMARY ===")     
                                              
checks = {                                                                                                     
    "Deliveries row count":         len(deliveries) == 279586,
    "All expected columns present": len(deliveries.columns) >= 33,                                             
    "No null batters":              deliveries["batter"].isnull().sum() == 0,
    "Over range valid (1-20)":      deliveries["over"].between(1, 20).all(),                                   
    "Phase values valid":           set(deliveries["phase"].unique()) == {"powerplay", "middle", "death"},     
    "Super over rows flagged":      deliveries[deliveries["innings"] > 2]["super_over"].all(),                 
    "No negative required_run_rate":deliveries["required_run_rate"].dropna().min() >= 0,                       
    "required_run_rate capped at 36":deliveries["required_run_rate"].dropna().max() <= 36,                     
    "No inf in run_rate_pressure":  not deliveries["run_rate_pressure"].isin([float("inf")]).any(),            
    "1st innings RRR all null":     deliveries[deliveries["innings"]==1]["required_run_rate"].isnull().all(),  
}                                                                                                              
                                                                                                                 
for check, passed in checks.items():                                                                           
    status = "PASS" if passed else "FAIL"                                                                      
    print(f"  [{status}] {check}")

=== VALIDATION SUMMARY ===
  [PASS] Deliveries row count
  [PASS] All expected columns present
  [PASS] No null batters
  [PASS] Over range valid (1-20)
  [PASS] Phase values valid
  [PASS] Super over rows flagged
  [PASS] No negative required_run_rate
  [PASS] required_run_rate capped at 36
  [PASS] No inf in run_rate_pressure
  [PASS] 1st innings RRR all null
